# 🎵 Music Hit or Flop Predictor - Machine Learning Pipeline
Dataset: Spotify Track Data | Features: Tempo, Loudness, Key, Mode, Energy

## 🛠️ Step 1: Preprocessing & Data Balancing
We filter for valid acoustic signatures (energy > 0.1) and establish a **Popularity Threshold of 15**. This threshold is specifically optimized to detect **Emerging Hits**, capturing local chart-toppers and indie tracks that mainstream metrics often overlook.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('data/dataset.csv')
features = ['tempo', 'loudness', 'key', 'mode', 'energy']
df = df.dropna(subset=features + ['popularity'])
df = df[df['energy'] > 0.1]

THRESHOLD = 15
df['is_hit'] = (df['popularity'] >= THRESHOLD).astype(int)

df_hits = df[df['is_hit'] == 1]
df_flops = df[df['is_hit'] == 0]
min_samples = min(len(df_hits), len(df_flops))
df_balanced = pd.concat([
    df_hits.sample(n=min_samples, random_state=42),
    df_flops.sample(n=min_samples, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'📊 Balanced Distribution: {min_samples} Hits & {min_samples} Flops.')

## 🧠 Step 2: Training 'The Big 5' Models (Voting Consensus)
We train 5 distinct algorithms to build a high-performance ensemble system: RandomForest, AdaBoost, KNN, DecisionTree, and XGBoost (GPU Accelerated).

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import joblib

X = df_balanced[features]
y = df_balanced['is_hit']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, 'feature_scaler.pkl')

models = {
    'RandomForest': RandomForestClassifier(n_estimators=100),
    'AdaBoost': AdaBoostClassifier(n_estimators=100),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'DecisionTree': DecisionTreeClassifier(max_depth=10),
    'XGBoost': XGBClassifier(tree_method='hist', device='cuda')
}

for name, model in models.items():
    try:
        model.fit(X_train_scaled, y_train)
    except:
        model.set_params(device='cpu')
        model.fit(X_train_scaled, y_train)
    
    acc = accuracy_score(y_test, model.predict(X_test_scaled))
    print(f'✅ Model {name} Trained! Accuracy: {acc:.4f}')
    joblib.dump(model, f'models/{name.lower()}_model.pkl')

## 🗳️ Step 3: Production Inference Logic
Core methodology for testing real-world audio samples using the collective consensus of all trained models.

In [ ]:
import os
from librosa_extractor import extract_features_from_audio

def final_prediction(audio_path):
    features = extract_features_from_audio(audio_path)
    X_input = np.array([[features[k] for k in features if k in ['tempo','loudness','key','mode','energy']]])
    X_scaled = scaler.transform(X_input)
    
    hit_count = 0
    for f in os.listdir('models'):
        m = joblib.load(os.path.join('models', f))
        if m.predict(X_scaled)[0] == 1: hit_count += 1
        
    print(f'Voting Result: {hit_count}/5 Models predict HIT!')